In [1]:
# Cell 1 - Get all article URLs from France 24 and save to json
# Site: https://www.france24.com/en/tag/2026-winter-olympics/
# Status: ALLOWED

import requests
from bs4 import BeautifulSoup
import time
import logging
import json

logging.basicConfig(
    filename="f24_get_webs.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True)

BASE_URL = "https://www.france24.com"
headers = {'User-Agent': 'MRU Data Science Student Project'}

def get_articles(soup) -> list:
    links = []
    logging.info('request success, getting urls')

    for a in soup.find_all('a', href=True):
        href = a['href']
        if '/en/' in href and '2026' in href:
            # Always store the full URL
            full_url = href if href.startswith('http') else BASE_URL + href
            if full_url not in links:
                links.append(full_url)

    print(f'Found {len(links)} article URLs')
    return links

def save_json(lists):
    data = {}
    for index, page_list in enumerate(lists):
        data[f'page{index + 1}'] = page_list
    logging.info('saving to json')
    with open('f24_url_data.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def main():
    f24_url = "https://www.france24.com/en/tag/2026-winter-olympics/"

    total_urls = 0
    each_page_lists = []

    for i in range(1, 6):
        this_page_url = f"{f24_url}?page={i}"

        try:
            logging.info(f'requesting {this_page_url}')
            response = requests.get(this_page_url, headers=headers)
            soup = BeautifulSoup(response.content, 'html.parser')

        except requests.exceptions.RequestException as e:
            print(f"error occurred: {e}")
            logging.info('failed requesting')
            continue

        except Exception as e:
            print(f"error occurred: {e}")
            logging.info('failed requesting')
            continue

        articles_list = get_articles(soup)
        each_page_lists.append(articles_list)
        total_urls += len(articles_list)

        time.sleep(3)

    print(f'total articles: {total_urls}')
    logging.info(f'Total URLs collected: {total_urls}')
    print('will save the urls to json...')
    save_json(each_page_lists)
    print('saved')

main()

Found 51 article URLs
Found 51 article URLs
Found 51 article URLs
Found 51 article URLs
Found 51 article URLs
total articles: 255
will save the urls to json...
saved


In [ ]:
# Cell 2 - Visit each URL, scrape the article, run analysis, save to JSON

import requests
from bs4 import BeautifulSoup
import time
import logging
import csv
import json
from datetime import datetime

headers = {'User-Agent': 'MRU Data Science Student Project'}

def web_scrapping(url, page_num, article_num):
    logging.basicConfig(
        filename='f24_scraping_log.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True)

    time.sleep(4)

    try:
        logging.info(f"Starting request for: {url}")
        # url is already the full URL from Cell 1 — do not prepend anything
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        article = {}
        article['Article_ID'] = f'P{page_num}A{article_num}'
        article['Source_ID'] = 'france24.com'
        article['Title'] = soup.h1.get_text(strip=True)
        article['Publish_Date'] = soup.find('time').get_text(strip=True)
        article['URL'] = url
        article['Article_Text'] = " ".join(p.get_text(strip=True) for p in soup.find_all('p') if p.get_text(strip=True))

        logging.info(f"Successfully scraped: {url}")
        return article

    except requests.exceptions.HTTPError as e:
        logging.error(f"HTTP Error for {url}: {e}")

    except requests.exceptions.ConnectionError as e:
        logging.error(f"Connection Error for {url}: {e}")

    except Exception as e:
        logging.error(f"Unexpected Error for {url}: {e}")

    return None

def analyze(data):
    target_keywords = {
    "Russia": ["Russia", "Russian", "Moscow", "AIN", "Neutral Athlete"],
    "Belarus": ["Belarus", "Belarusian", "Minsk"],
    "Israel": ["Israel", "Israeli", "Tel Aviv"],
    "China": ["China", "Chinese", "Beijing"],
    "North Korea": ["North Korea", "DPRK", "Korean"],
    "Iran": ["Iran", "Iranian", "Tehran"],
    "Ukraine": ["Ukraine", "Ukrainian", "Kyiv"]
}
    political_keywords = ["war", "invasion", "sanction", "protest", "ban", "neutrality", "conflict"]

    url = data['URL']
    title = data['Title']
    content_to_check = (url + " " + title).lower()

    found_countries = ''
    for country, keywords in target_keywords.items():
        if any(kw.lower() in content_to_check for kw in keywords):
            found_countries += f'{country},'

    if found_countries == '':
        return None

    text = data['Article_Text'].lower()
    mention_count = sum(text.count(kw.lower()) for kw in target_keywords)
    has_political_context = sum(text.count(p_kw.lower()) for p_kw in political_keywords)

    if mention_count >= 2 and has_political_context:
        framing_analysis = {}
        framing_analysis['Article_ID'] = data['Article_ID']
        framing_analysis['Target_Nation'] = found_countries[:-1]
        framing_analysis['Frequency_of_Political_Terms'] = has_political_context
        return framing_analysis

    return None

def update_acquisition_log(domain, records_count, errors_count, related_count):
    log_file = 'acquisition_log.csv'
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(log_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([timestamp, domain, records_count, errors_count, related_count])

def save_to_json(all_articles, analysis):
    with open("f24_Articles.json", "w") as f:
        json.dump(all_articles, f, indent=4)
    with open("f24_Source_ID.json", "w") as f:
        json.dump(analysis, f, indent=4)
    print(f"Saved {len(all_articles)} total articles to f24_Articles.json")
    print(f"Saved {len(analysis)} politically relevant articles to f24_Source_ID.json")

def main():
    with open('f24_url_data.json', 'r', encoding='utf-8') as f:
        page_urls = json.load(f)

    success_count = 0
    errors_count = 0
    related_count = 0
    all_articles = []
    framing_analysis_ = []

    for page in page_urls:
        for index, url in enumerate(page_urls[page], start=1):
            # url is already the full URL — use it directly
            result = web_scrapping(url, page[4:], index)

            if result is None:
                errors_count += 1
                continue

            success_count += 1
            all_articles.append(result)  # save every successful scrape

            analysis = analyze(result)
            if analysis is not None:
                related_count += 1
                framing_analysis_.append(analysis)

    save_to_json(all_articles, framing_analysis_)
    update_acquisition_log("france24.com", success_count, errors_count, related_count)

main()

Saved 255 total articles to f24_Articles.json
Saved 35 politically relevant articles to f24_Source_ID.json
